# House Stage 2: Monte Carlo

Stage 1 (`house.stage1`) gives a point prediction of each district's two-party margin (D − R). Stage 2 turns those into a distribution of outcomes by simulating the errors.

1. **Stage 1 inputs:** the rolling out-of-sample predictions and residuals for 2018–2024, and the 2026 forecast.
2. **Error model:** the structure of the errors and its parameters, estimated from the residuals. *(yours)*
3. **Simulation:** draw errors and add them to the predictions. *(yours)*
4. **Calibration backtest:** simulate each past eval year from an error model estimated on earlier years only, and check coverage. *(yours)*
5. **2026 outputs:** district win probabilities and the seat distribution. *(yours)*

Residuals are actual − predicted, in D − R points; positive means Democrats outperformed Stage 1.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # for experiments.house
import house.regression as hr

In [ ]:
from house import stage1

CONFIG = stage1.HOUSE_ENSEMBLE_AB_85_15

In [ ]:
SEED = 2026         # your choice
N_SIMS = 10_000     # your choice
rng = np.random.default_rng(SEED)

with warnings.catch_warnings(action="ignore"):  # loading warns about known optional-column gaps
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, 2006-2026
    targets = hr.load_targets()        # data/processed/house/targets.csv, 2006-2024

## 1. Stage 1 inputs

`rolling[year]` is fit on 2006 through the election before `year`, so its residuals are out of sample. `oos` stacks them, one row per scored district-year. The competitive flags come from Stage 1's own scoring (ex-ante: |lean + generic ballot| ≤ 10; ex-post: |actual margin| ≤ 10, the `config.py` default).

In [ ]:
with warnings.catch_warnings(action="ignore"):
    rolling = stage1.rolling_predictions(CONFIG, predictors=predictors, targets=targets)

flags = ["competitive_ex_ante", "competitive_ex_post"]
oos = pd.concat([r.predictions.merge(r.scored[[*hr.KEY, *flags]], on=hr.KEY, how="left", validate="one_to_one")
                 for r in rolling.values()], ignore_index=True)
oos = oos.merge(predictors[[*hr.KEY, "partisan_lean"]], on=hr.KEY, how="left", validate="one_to_one")
oos = oos[oos["actual"].notna()]  # every row here has an observed target; kept explicit
oos.groupby("year").size().rename("districts")

In [ ]:
# What the error model has to work with: each year's national error (mean) and the district spread around it (sd)
oos.groupby("year")["residual"].agg(n="size", mean="mean", sd="std").round(2)

### The 2026 forecast

This raises until 2026's `gen_margin_raw` is in `data/processed/house/modeling.csv`. Every 2026 district is predicted, including ones expected to be uncontested.

In [ ]:
with warnings.catch_warnings(action="ignore"):
    forecast = stage1.forecast(CONFIG, predictors=predictors, targets=targets)
point = forecast.predictions  # year, state, district, prediction (actual and residual are blank)
point.head()

## 2. Error model *(yours)*

Decide the error structure and estimate its parameters from `oos`, for example which components (national, state, district) there are, their distributions, and whether the spread depends on competitiveness. Keep each estimate in a named variable so the simulation cell reads them directly.

Only 4 eval years inform any national component, so its variance is poorly determined.

In [ ]:
# TODO: estimate the error-model parameters from `oos`


## 3. Simulation *(yours)*

Target output: `sims`, an array of shape `(N_SIMS, len(point))` of simulated margins, with columns in the row order of `point`. Use `rng` for every draw so runs are reproducible.

In [ ]:
# TODO: sims = point["prediction"].to_numpy() + <simulated errors>


## 4. Calibration backtest *(yours)*

For each eval year in `rolling`, estimate the error model from the *earlier* eval years' residuals only, simulate that year from `rolling[year].predictions`, and compare with its actual results: interval coverage, Brier score of the win probabilities, and where the actual seat count falls in the simulated distribution. 2018 has no earlier eval year, so decide how to handle it.

In [ ]:
# TODO: calibration loop over rolling


## 5. 2026 outputs *(yours)*

From `sims`: each district's probability of a Democratic win, the distribution of Democratic seats out of 435, and its median and intervals.

In [ ]:
# TODO: win probabilities and seat distribution from sims
